# Human-to-NAO Upper-Body Pose Imitation System  
**Vision-Based Upper-Body Pose Retargeting for a NAO Humanoid Robot**

---

## 1. Introduction

Human–robot interaction (HRI) increasingly relies on intuitive and natural communication channels. Among them, **human motion imitation** is a key capability for humanoid robots, enabling demonstrations, social interaction, and teleoperation.

This project presents a **real-time vision-based upper-body pose imitation system** for a NAO humanoid robot. Using a single RGB camera and a lightweight pose estimation model, the robot is able to imitate human head, arm, and torso motions in real time, while maintaining balance and safety.

---

## 2. System Overview

The system follows a **client–server architecture**, separating perception and robot control to ensure modularity and real-time performance.
```java
Human → RGB Camera → Pose Client (Windows)
                            |
                            |  UDP (JSON)
                            ↓
                   Pose Server (WSL / Linux)
                            |
                            ↓
                       NAO Robot
```


### Main Components

| Component | Description |
|----------|-------------|
| Pose Client | Human pose detection and joint angle computation |
| UDP Communication | Low-latency transmission of pose data |
| Pose Server | NAO joint control and balance management |
| NAO Robot | Physical execution of the imitation |

---

## 3. Human Pose Estimation

### 3.1 Pose Detection

- **Framework:** MediaPipe Pose  
- **Input:** Monocular RGB camera  
- **Output:** 33 human body landmarks (head, shoulders, elbows, wrists, hips, etc.)

MediaPipe Pose was selected due to its:
- Real-time performance on CPU
- Robustness to noise
- Ease of deployment without calibration

---

### 3.2 Coordinate System Considerations

MediaPipe uses a **camera-centered coordinate system**:

- x-axis: left–right  
- y-axis: up–down  
- z-axis: forward–backward (depth)

In contrast, NAO joints operate in a **robot-centric joint space**.  
Therefore, direct mapping is not possible, and intermediate vector-based representations and geometric reasoning are required.

---

## 4. Human-to-NAO Motion Mapping

### 4.1 Head Motion

Human head motion is estimated using the vector from the shoulder midpoint to the nose.

| Human Motion | NAO Joint |
|-------------|-----------|
| Head rotation (left/right) | HeadYaw |
| Head tilt (up/down) | HeadPitch |

Gain factors are applied to compensate for the limited motion amplitude of the NAO head.

---

### 4.2 Shoulder Motion (Key Challenge)

#### Shoulder Pitch (Arm Raise / Lower)

- Computed from the angle between the upper arm vector and the vertical downward direction
- Enables:
  - Natural arm-down posture
  - Forward arm raise
  - Overhead arm raise (within NAO limits)

#### Shoulder Roll (Side Raise / T-pose)

A major challenge was distinguishing between:
- **Forward arm raise** (hands in front of the chest)
- **Side arm raise** (T-pose)

A naive x-axis based mapping caused incorrect T-pose activation.  
To solve this, an **intention-aware strategy** was introduced:

- If `|x| > |z|` → side raise  
- If `|z| > |x|` → forward raise (suppress roll)

This significantly improved semantic correctness of the imitation.

---

### 4.3 Elbow Motion

Elbow flexion is computed from the angle between:
- Upper arm (elbow → shoulder)
- Forearm (elbow → wrist)

The resulting flexion angle is mapped to NAO joint limits:

- `LElbowRoll ∈ [−1.5, −0.03]`
- `RElbowRoll ∈ [0.03, 1.5]`

To improve motion quality:
- Small-angle dead zones were introduced
- Motion smoothing was applied
- Joint saturation prevents unsafe bending

---

### 4.4 Torso Inclination (Safe Approximation)

NAO does not allow direct control of the torso joint.  
Instead, torso inclination is **approximated via hip joints**:

- Torso roll → `LHipRoll`, `RHipRoll`
- Torso pitch → `LHipPitch`, `RHipPitch`

All torso-related motions are:
- Strictly limited in amplitude
- Executed under the Whole Body Balancer

---

## 5. Real-Time Stability and Safety

### 5.1 Motion Smoothing

An exponential moving average filter is applied to all joint angles:

\[
\theta_t = \alpha \cdot \theta_{t-1} + (1-\alpha)\cdot \theta_t
\]

This reduces jitter, improves smoothness, and avoids abrupt joint movements.

---

### 5.2 Balance Control

- NAO’s **WholeBodyBalancer** is enabled
- The robot remains in a double-support stance
- Joint speeds and ranges are constrained

This ensures stable operation even when the human performs fast motions.

---

## 6. Experimental Results and Observations

### 6.1 Achievements

- Real-time head imitation  
- Correct arm raising and lowering  
- Natural elbow bending  
- Controlled T-pose activation  
- Forward arm raise without unintended side expansion  
- Stable standing posture  

---

### 6.2 Challenges and Solutions

| Issue | Cause | Solution |
|------|------|----------|
| Mirrored arm motion | Coordinate mismatch | Explicit direction correction |
| Locked elbow motion | Incorrect angle mapping | Redefined elbow flexion model |
| Unintended T-pose | x-axis dominance | x–z intention discrimination |
| Small motion amplitude | Low gain | Joint-specific gain tuning |
| Loss of balance | Excessive motion | Range limiting and balancing |

---

## 7. Limitations

- No inverse kinematics solver is used  
- No force or torque feedback  
- Wrist and finger motions are not modeled  
- Only single-person, frontal-view interaction is supported  

---

## 8. Conclusion

This project demonstrates a **vision-based real-time upper-body pose imitation system** for a NAO humanoid robot.  
By carefully designing geometric mappings, intention-aware heuristics, and safety constraints, the robot is able to imitate human motions in a natural and stable manner.

The system provides a solid foundation for future research in human–robot interaction, teleoperation, and imitation learning.
